##**Overview Of Project**

**Edge AI Deployment IDE**

The goal of this project was to build a practical, hardware-aware dashboard that helps estimate if a ML/DL model will actually fit on a specific Edge device (like a Jetson Nano or Raspberry Pi) before spending hours trying to deploy it. This tool is built to handle both Classical ML and Deep Learning, as edge devices usually have both a CPU and an NPU/GPU.

---


1.   The Classical ML Branch (CPU): Estimates inference operations and SRAM usage for models like Random Forests and Decision Trees. I've added dynamic Big O calculations to show how memory explodes exponentially ($2^{depth}$) if a tree gets too deep.
2.   The Deep Learning Branch (NPU): Uses PyTorch (thop and torchinfo) to trace network graphs and extract MACs/FLOPs. It applies the Roofline Model to check if a CNN is memory-bound or compute-bound.

## README.md

```markdown
# Edge AI Deployment IDE

## Overview
This project provides a practical, hardware-aware dashboard to estimate if a Machine Learning/Deep Learning model will fit on a specific Edge device (like a Jetson Nano or Raspberry Pi) before deployment. It supports both Classical ML and Deep Learning models, leveraging both CPU and NPU/GPU capabilities of edge devices.

### Key Features:
-   **Classical ML Branch (CPU):** Estimates inference operations and SRAM usage for models like Random Forests and Decision Trees. Includes dynamic Big O calculations to show memory growth.
-   **Deep Learning Branch (NPU):** Uses PyTorch (`thop` and `torchinfo`) to trace network graphs, extract MACs/FLOPs, and apply the Roofline Model to determine if a CNN is memory-bound or compute-bound.

## Prerequisites
-   A ngrok account is required. Please use your own Auth Token in the last cell of this notebook.

## Installation

1.  **Install Dependencies:**
    Run the following command in a code cell to install the necessary Python libraries and npm package:
    ```bash
    !pip install -q streamlit thop torchinfo onnx pyngrok fpdf2 reportlab matplotlib

    !npm install -g localtunnel
    ```

2.  **Core IDE Engine (`app.py`):**
    The Streamlit application's core logic is contained in `app.py`. This file is generated by a notebook cell. Ensure this cell is run to create the `app.py` file.

## Usage

1.  **Stop Previous Background Tasks:**
    Before running the app, execute the cell that stops any previous `streamlit` or `ngrok` processes to prevent conflicts:
    ```bash
    !pkill -f streamlit
    !pkill -f ngrok
    from pyngrok import ngrok
    ngrok.kill()
    ```

2.  **Public Deployment:**
    Replace the placeholder `NGROK_TOKEN` with your actual ngrok authentication token and run the cell. This will launch the Streamlit app and provide a public URL to access it.
    ```python
    from pyngrok import ngrok
    NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN"
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8501)
    print(f" Click this link to open your IDE: {public_url}")
    !streamlit run app.py &>/content/logs.txt &
    ```

3.  **Interact with the IDE:**
    Open the provided public URL in your browser. You can then select your target edge architecture (Deep Learning or Traditional ML), define hardware capabilities and project constraints, and run simulations to evaluate your model's deployment feasibility.

Enjoy deploying your AI models to the edge with confidence!
```

Edge AI Hardware-Aware Deployment IDE:
  1. Quantization simulation (FP32, FP16, INT8, INT4)
  2. Cache-miss latency model for CPU branch (L1/L2/DRAM penalties)
  3. Real ONNX file parsing & shape inference
  4. Granular Spatial architecture inspection
  5. Roofline Matplotlib Visualization
  6. ReportLab PDF export of PPA scorecard


##**Step 1: Install Dependencies**

In [1]:
# Install required libraries for Deep Learning profiling and UI
!pip install -q streamlit thop torchinfo onnx pyngrok fpdf2 reportlab matplotlib
!npm install -g localtunnel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 91.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

##**Step 2: Core IDE Engine (app.py)**

In [5]:
%%writefile app.py

"""
Edge AI Hardware-Aware Deployment IDE v2
Enhanced with:
  1. Quantization simulation (FP32, FP16, INT8, INT4)
  2. Cache-miss latency model for CPU branch (L1/L2/DRAM penalties)
  3. Real ONNX file parsing & shape inference
  4. Granular Spatial architecture inspection
  5. Roofline Matplotlib Visualization
  6. ReportLab PDF export of PPA scorecard
  7. Detailed Actionable MLOps Diagnostics
"""

import streamlit as st
import torch
from torchvision import models
from thop import profile
from torchinfo import summary
import pandas as pd
import numpy as np
import onnx
from onnx import shape_inference
import matplotlib.pyplot as plt
from datetime import datetime
import io
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# ==========================================
# MODULE 1: UI INITIALIZATION & FORMATTING
# ==========================================

st.set_page_config(layout="wide", page_title="Edge AI Deployment IDE v2")

st.markdown("""
<style>
[data-testid="stMetricValue"] {
    font-size: 1.8rem !important;
    white-space: normal !important;
    word-wrap: break-word !important;
}
[data-testid="stMetricLabel"] {
    white-space: normal !important;
}
</style>
""", unsafe_allow_html=True)

# ==========================================
# MODULE 2: CONFIGURATION & HELPERS
# ==========================================

QUANTIZATION_CONFIG = {
    "FP32": {"mem_mul": 1.0, "lat_mul": 1.0, "bytes": 4, "acc_loss": "none"},
    "FP16": {"mem_mul": 0.5, "lat_mul": 0.6, "bytes": 2, "acc_loss": "~0.1%"},
    "INT8": {"mem_mul": 0.25, "lat_mul": 0.35, "bytes": 1, "acc_loss": "~0.5–1%"},
    "INT4": {"mem_mul": 0.125, "lat_mul": 0.22, "bytes": 0.5, "acc_loss": "~1–3%"},
}

def format_memory(mb_value):
    if mb_value >= 1_048_576: return f"{mb_value / 1_048_576:,.2f} TB"
    elif mb_value >= 1024: return f"{mb_value / 1024:,.2f} GB"
    return f"{mb_value:,.1f} MB"

def format_power(w_value):
    if w_value >= 1000: return f"{w_value / 1000:,.2f} kW"
    return f"{w_value:,.2f} W"

def format_area(mm2_value):
    if mm2_value >= 1_000_000: return f"{mm2_value / 1_000_000:,.2f} m²"
    return f"{mm2_value:,.1f} mm²"

def cache_miss_latency(total_mb, infer_ops, clk_mhz, l1_kb, l2_kb):
    model_bytes = total_mb * 1024 * 1024
    l1_bytes = l1_kb * 1024
    l2_bytes = l2_kb * 1024
    clk_hz = clk_mhz * 1e6

    l1_cycles_penalty = 4
    l2_cycles_penalty = 12
    dram_cycles_penalty = 200

    if model_bytes > l2_bytes:
        dram_fraction = min((model_bytes - l2_bytes) / model_bytes, 0.8)
        miss_penalty_cycles = infer_ops * dram_cycles_penalty * dram_fraction * 0.3
        cache_hit_level = "DRAM"
    elif model_bytes > l1_bytes:
        l2_fraction = min((model_bytes - l1_bytes) / model_bytes, 0.9)
        miss_penalty_cycles = infer_ops * l2_cycles_penalty * l2_fraction * 0.2
        cache_hit_level = "L2"
    else:
        miss_penalty_cycles = infer_ops * l1_cycles_penalty * 0.05
        cache_hit_level = "L1 (hot)"

    base_lat_s = infer_ops / clk_hz
    miss_lat_s = miss_penalty_cycles / clk_hz
    total_lat_s = (base_lat_s + miss_lat_s) * 10
    miss_penalty_pct = (miss_lat_s / (base_lat_s + miss_lat_s) * 100) if (base_lat_s + miss_lat_s) > 0 else 0

    return {"latency_s": total_lat_s, "cache_hit_level": cache_hit_level, "miss_penalty_pct": miss_penalty_pct}

def generate_pdf_scorecard(result_data):
    pdf_buffer = io.BytesIO()
    doc = SimpleDocTemplate(pdf_buffer, pagesize=letter, topMargin=0.5*inch, bottomMargin=0.5*inch)
    story = []
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle("CustomTitle", parent=styles["Heading1"], fontSize=18, textColor=colors.HexColor("#1a1a1a"), spaceAfter=6)
    subtitle_style = ParagraphStyle("CustomSubtitle", parent=styles["Normal"], fontSize=10, textColor=colors.HexColor("#666666"), spaceAfter=12)

    story.append(Paragraph("Edge AI PPA Scorecard", title_style))
    story.append(Paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", subtitle_style))

    hw_info = result_data.get("hardware", {})
    lim_info = result_data.get("limits", {})

    hw_text = (
        f"<b>Hardware:</b> {hw_info.get('tops', 'N/A')} TOPS • {hw_info.get('bw', 'N/A')} GB/s BW • "
        f"{hw_info.get('clk', 'N/A')} MHz • {hw_info.get('pwr', 'N/A')} W base • L1 {hw_info.get('l1', 'N/A')} KB • L2 {hw_info.get('l2', 'N/A')} KB<br/>"
        f"<b>Constraints:</b> Memory ≤{lim_info.get('mem', 'N/A')} MB • Latency ≤{lim_info.get('lat', 'N/A')} ms • "
        f"Power ≤{lim_info.get('pwr', 'N/A')} W • Area ≤{lim_info.get('area', 'N/A')} mm²"
    )
    story.append(Paragraph(hw_text, styles["Normal"]))
    story.append(Spacer(1, 0.3*inch))

    metrics = result_data.get("metrics", [])
    data = [["Metric", "Value", "Constraint Check"]]

    for metric in metrics:
        status = metric.get("status", "N/A")
        status_color = colors.HexColor("#2a7a3b") if status == "PASS" else colors.HexColor("#c0392b") if status == "FAIL" else colors.HexColor("#3498db")
        data.append([metric.get("name", ""), metric.get("value", ""), Paragraph(f"<font color={status_color.hexval()}><b>{status}</b></font>", styles["Normal"])])

    table = Table(data, colWidths=[2.5*inch, 1.8*inch, 1.5*inch])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#f5f5f3")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("ALIGN", (0, 0), (-1, -1), "LEFT"),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("BOTTOMPADDING", (0, 0), (-1, 0), 8),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#e0e0e0")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#fafafa")]),
    ]))
    story.append(table)
    story.append(Spacer(1, 0.3*inch))

    verdict = result_data.get("verdict", "N/A")
    verdict_color = colors.HexColor("#2a7a3b") if "APPROVED" in verdict else colors.HexColor("#c0392b")
    story.append(Paragraph(f"<b><font color={verdict_color.hexval()}>{verdict}</font></b>", styles["Normal"]))

    doc.build(story)
    pdf_buffer.seek(0)
    return pdf_buffer

# ==========================================
# MODULE 3: HARDWARE SIDEBAR & UI
# ==========================================

st.title("Edge AI Hardware-Aware IDE v2")
st.markdown("PPA & CPU/NPU Routing Dashboard — with Quantization, Cache Modeling & Spatial Inspection")

st.sidebar.header("1. Hardware Capabilities")
hw_tops = st.sidebar.number_input("Peak Compute (TOPS/GFLOPS)", value=1.0, min_value=0.01, step=0.1)
hw_bw = st.sidebar.number_input("Memory Bandwidth (GB/s)", value=4.0, min_value=0.1, step=0.5)
hw_clock = st.sidebar.number_input("CPU/NPU Clock Speed (MHz)", value=1000, min_value=10, step=100)
hw_power_base = st.sidebar.number_input("Device Base Power (Watts)", value=2.0, min_value=0.1, step=0.5)
hw_l1_cache = st.sidebar.number_input("L1 Cache (KB)", value=64, min_value=16, step=16)
hw_l2_cache = st.sidebar.number_input("L2 Cache (KB)", value=512, min_value=128, step=128)

st.sidebar.header("2. Project Constraints (PPA)")
max_memory_mb = st.sidebar.number_input("Max Memory Allowed (MB)", value=30.0, step=5.0)
max_latency_ms = st.sidebar.number_input("Max Latency Allowed (ms)", value=50.0, step=10.0)
max_power_w = st.sidebar.number_input("Max Power Allowed (Watts)", value=5.0, step=1.0)
max_area_mm2 = st.sidebar.number_input("Max Silicon Area (mm²)", value=40.0, step=5.0)

st.subheader("3. Select Software Workload & Target Silicon")

ml_paradigm = st.radio(
    "Select Target Edge Architecture:",
    ["Deep Learning (Edge NPU / GPU)", "Traditional ML (Edge CPU)"],
    horizontal=True,
)

# ==========================================
# BRANCH A: DEEP LEARNING (NPU/GPU TARGET)
# ==========================================

if ml_paradigm == "Deep Learning (Edge NPU / GPU)":
    st.info("💡 **Routing Workload to NPU/GPU:** Deep Learning models rely on dense matrix multiplication. Mapped to Edge NPU/GPU with Roofline analysis.")

    source_type = st.radio("Choose Model Source:", ["Select Pre-loaded Edge Model", "Upload ONNX File", "Custom .pth File"], horizontal=True)

    model_ready = False
    gflops = 0.0
    total_memory_mb = 0.0
    macs = 0.0
    params = 0.0
    onnx_ops_df = None
    model_name = ""
    display_name = ""

    if source_type == "Select Pre-loaded Edge Model":
        model_name = st.selectbox("Choose a baseline architecture:", [
            "MobileNetV2 (Lightweight Vision)", "MobileNetV3 Small (Highly Edge Optimized)",
            "ShuffleNet V2 (Ultra-Lightweight)", "SqueezeNet 1.1 (High Compression)",
            "ResNet18 (Standard Vision)", "AlexNet (Legacy Heavy Vision)"
        ])

        @st.cache_resource
        def load_model(name):
            if "MobileNetV2" in name: return models.mobilenet_v2()
            elif "MobileNetV3 Small" in name: return models.mobilenet_v3_small()
            elif "ShuffleNet" in name: return models.shufflenet_v2_x1_0()
            elif "SqueezeNet" in name: return models.squeezenet1_1()
            elif "ResNet18" in name: return models.resnet18()
            elif "AlexNet" in name: return models.alexnet()

        model = load_model(model_name)
        model_ready = True
        display_name = model_name.split(" ")[0]

    elif source_type == "Upload ONNX File":
        uploaded_file = st.file_uploader("Upload .onnx file", type=["onnx"])
        if uploaded_file is not None:
            display_name = uploaded_file.name
            try:
                onnx_model = onnx.load_model_from_string(uploaded_file.getvalue())
                params = 0
                for initializer in onnx_model.graph.initializer:
                    layer_params = 1
                    for dim in initializer.dims: layer_params *= dim
                    params += layer_params

                weight_memory_mb = (params * 4) / (1024 * 1024)
                total_memory_mb = weight_memory_mb * 1.2

                with st.spinner("Inferring Tensor Shapes and Walking Graph..."):
                    inferred_model = shape_inference.infer_shapes(onnx_model)
                    shape_dict = {vi.name: [dim.dim_value for dim in vi.type.tensor_type.shape.dim] for vi in list(inferred_model.graph.value_info) + list(inferred_model.graph.input) + list(inferred_model.graph.output) if vi.type.tensor_type.HasField("shape")}

                    calculated_onnx_macs = 0
                    op_counts = {}
                    for node in inferred_model.graph.node:
                        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
                        try:
                            if node.op_type == "Conv":
                                in_shape = shape_dict.get(node.input[0], [])
                                out_shape = shape_dict.get(node.output[0], [])
                                k_h, k_w = 3, 3
                                for attr in node.attribute:
                                    if attr.name == 'kernel_shape': k_h, k_w = attr.ints[0], attr.ints[1]
                                if len(in_shape) >= 4 and len(out_shape) >= 4:
                                    calculated_onnx_macs += (out_shape[2] * out_shape[3] * out_shape[1] * in_shape[1] * k_h * k_w)
                            elif node.op_type in ["Gemm", "MatMul"]:
                                in_shape = shape_dict.get(node.input[0], [])
                                out_shape = shape_dict.get(node.output[0], [])
                                if len(in_shape) >= 2 and len(out_shape) >= 2:
                                    calculated_onnx_macs += (in_shape[-1] * out_shape[-1])
                        except: pass

                onnx_ops_df = pd.DataFrame(list(op_counts.items()), columns=["ONNX Operation Type", "Layer Count"]).sort_values(by="Layer Count", ascending=False).reset_index(drop=True)

                if calculated_onnx_macs > 0:
                    gflops = (calculated_onnx_macs * 2) / 1e9
                    macs = calculated_onnx_macs
                    st.success(f"✓ Parsed Graph: {params:,} parameters | Estimated {gflops:.3f} GFLOPs")
                else:
                    gflops = st.number_input("Estimated GFLOPs (Auto-calculate failed)", value=0.5, step=0.1)
                    macs = (gflops * 1e9) / 2
                model_ready = True
            except Exception as e:
                st.error(f"Error parsing ONNX: {e}")

    elif source_type == "Custom .pth File":
        uploaded_file = st.file_uploader("Upload .pth file", type=["pth"])
        if uploaded_file is not None:
            file_size_mb = uploaded_file.size / (1024 * 1024)
            st.success(f"✓ PyTorch file loaded ({file_size_mb:.1f} MB)")
            col1, col2 = st.columns(2)
            gflops_manual = col1.number_input("Estimated GFLOPs", value=0.5, min_value=0.01, step=0.1)
            total_memory_mb = file_size_mb * 1.2
            params = (file_size_mb * 1024 * 1024) / 4
            gflops = gflops_manual
            macs = (gflops * 1e9) / 2
            model_ready = True
            display_name = f"Custom .pth"

    st.markdown("**Precision & Quantization:**")
    quant_mode = st.radio("Select precision level:", ["FP32 (Baseline)", "FP16 (Half Precision)", "INT8 (Post-Training Quantization)", "INT4 (Extreme Compression)"], horizontal=True)
    quant_key = quant_mode.split(" ")[0]
    quant_cfg = QUANTIZATION_CONFIG[quant_key]

    if st.button("Run Deep Learning Simulation", key="dl_sim"):
        if model_ready:
            with st.spinner("Calculating advanced PPA metrics..."):
                if source_type == "Select Pre-loaded Edge Model":
                    dummy_input = torch.randn(1, 3, 224, 224)
                    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
                    flops = macs * 2
                    gflops = flops / 1e9
                    weight_memory_mb = (params * 4) / (1024 * 1024)
                    total_memory_mb = weight_memory_mb * 1.2
                else:
                    flops = gflops * 1e9

                base_memory = total_memory_mb
                total_memory_mb *= quant_cfg["mem_mul"]
                est_area_mm2 = (total_memory_mb * 1.2) + (gflops * 0.8)
                arithmetic_intensity = flops / (total_memory_mb * 1024 * 1024)
                hw_ridge_point = (hw_tops * 1e12) / (hw_bw * 1e9)
                is_memory_bound = arithmetic_intensity < hw_ridge_point

                if is_memory_bound: latency_seconds = (total_memory_mb / 1024) / hw_bw
                else: latency_seconds = gflops / (hw_tops * 1000)

                latency_seconds *= 1.5 * quant_cfg["lat_mul"]
                latency_ms = latency_seconds * 1000
                throughput_fps = 1000 / latency_ms if latency_ms > 0 else 0
                clock_cycles = latency_seconds * (hw_clock * 1e6)
                achieved_tops = (flops / latency_seconds) / 1e12 if latency_seconds > 0 else 0
                power_w = hw_power_base + (0.15 * (total_memory_mb / 50)) + (0.5 * (gflops / hw_tops)) * quant_cfg["lat_mul"]

                st.divider()
                st.markdown(f"### 📊 Deep Learning Analysis: `{display_name}` ({quant_key})")

                if quant_key != "FP32":
                    mem_saving = (1 - quant_cfg["mem_mul"]) * 100
                    lat_saving = (1 - quant_cfg["lat_mul"]) * 100
                    col1, col2, col3, col4 = st.columns(4)
                    col1.metric("Memory Saving", f"-{mem_saving:.0f}%", delta=f"From {base_memory:.1f} MB")
                    col2.metric("Latency Saving", f"-{lat_saving:.0f}%", delta=f"Accuracy loss: {quant_cfg['acc_loss']}")
                    col3.metric("Original Memory", f"{base_memory:.1f} MB")
                    col4.metric("Quantized Precision", quant_key)

                st.markdown("#### 🧠 1. Workload Profiling")
                c1, c2, c3, c4 = st.columns(4)
                c1.metric("Total MACs", f"{macs/1e6:,.1f} M")
                c2.metric("Total Compute (FLOPs)", f"{gflops:,.2f} GFLOPs")
                c3.metric("Arithmetic Intensity", f"{arithmetic_intensity:,.1f} FLOP/B")
                c4.metric("Bottleneck Type", "Memory" if is_memory_bound else "Compute")

                st.markdown("#### 💾 2. Physical Footprint (Area & Memory)")
                c1, c2, c3 = st.columns(3)
                c1.metric("Model Parameters", f"{params/1e6:,.2f} M")
                c2.metric("Memory Required", f"{total_memory_mb:,.2f} MB", delta=f"Limit: {max_memory_mb}", delta_color="inverse" if total_memory_mb > max_memory_mb else "normal")
                c3.metric("Silicon Area (Est)", f"{est_area_mm2:,.1f} mm²", delta=f"Limit: {max_area_mm2}", delta_color="inverse" if est_area_mm2 > max_area_mm2 else "normal")

                st.markdown("#### ⏱️ 3. Execution Metrics")
                c1, c2, c3, c4 = st.columns(4)
                c1.metric("Latency", f"{latency_ms:,.1f} ms", delta=f"Limit: {max_latency_ms}", delta_color="inverse" if latency_ms > max_latency_ms else "normal")
                c2.metric("Inference / Sec", f"{throughput_fps:,.1f} FPS")
                c3.metric("Clock Cycles", f"{clock_cycles/1e6:,.2f} M")
                c4.metric("Power Draw", f"{power_w:,.2f} W", delta=f"Limit: {max_power_w}", delta_color="inverse" if power_w > max_power_w else "normal")

                # RESTORED: Deep Learning Actionable Optimization Engine
                st.divider()
                st.markdown("### 🛠️ Hardware-Aware Optimization Engine")
                issues = 0

                if is_memory_bound:
                    st.warning(f"**Diagnostic:** Model is **Memory-Bound**. Arithmetic Intensity ({arithmetic_intensity:.1f}) is lower than the Hardware Ridge Point ({hw_ridge_point:.1f}).")
                    if quant_key == "FP32":
                        st.info("💡 **Action:** Apply **INT8 Post-Training Quantization** (using the toggle above). Compute optimization like pruning will yield diminishing returns here.")
                    else:
                        st.info(f"💡 **Action:** Already on {quant_key}. Consider **Knowledge Distillation** or **Structural Pruning** to further reduce payload size before it hits the memory bus.")
                else:
                    st.info(f"**Diagnostic:** Model is **Compute-Bound**. Utilizing {achieved_tops:.2f} TOPS during active inference.")
                    st.warning("💡 **Action:** Apply **Filter Pruning** or shift to a **Depthwise-Separable** architecture (like MobileNet) to drastically reduce the MAC count.")

                if total_memory_mb > max_memory_mb or est_area_mm2 > max_area_mm2:
                    st.error(f"🚨 **Physical Check Failed:** Model exceeds allocated Memory ({max_memory_mb} MB) or Area limits.")
                    st.info("💡 **Action:** You must quantize to a lower precision (INT8/INT4) or switch to a significantly smaller architecture to fit this chip.")
                    issues += 1

                if latency_ms > max_latency_ms or power_w > max_power_w:
                    st.error(f"🚨 **Execution Check Failed:** Model exceeds Latency ({max_latency_ms} ms) or Power constraints.")
                    st.info("💡 **Action:** If battery-powered, apply Dynamic Voltage and Frequency Scaling (DVFS) to the NPU, or implement early-exit neural branches to stop inference sooner.")
                    issues += 1

                if issues == 0:
                    st.success("✅ **Deployment Approved:** The architecture completely satisfies the target hardware PPA constraints.")
                else:
                    st.error(f"🚨 **Constraints Violated:** Failed {issues} hardware checks.")

                st.divider()
                st.markdown("### 📈 Roofline Model Visualization")
                fig, ax = plt.subplots(figsize=(10, 6))
                ai_values = np.logspace(-2, 3, 100)
                peak_performance = np.minimum(hw_tops * 1000, hw_bw * ai_values)
                ax.loglog(ai_values, peak_performance, "k-", linewidth=2, label="Roofline Bound")
                ax.axvline(hw_ridge_point, color="orange", linestyle="--", linewidth=1.5, label=f"Ridge Point ({hw_ridge_point:.1f})")
                model_performance = hw_bw * arithmetic_intensity if is_memory_bound else hw_tops * 1000
                ax.scatter(arithmetic_intensity, model_performance, s=200, c="red" if is_memory_bound else "green", marker="o", label="Model Operating Point", zorder=5)
                ax.set_xlabel("Arithmetic Intensity (FLOP/Byte)", fontsize=12)
                ax.set_ylabel("Performance (GFLOPS)", fontsize=12)
                ax.set_title(f"Roofline Analysis: {display_name} ({quant_key})", fontsize=14)
                ax.legend(loc="upper left")
                ax.grid(True, which="both", ls="-", alpha=0.2)
                st.pyplot(fig)

                st.divider()
                if source_type == "Select Pre-loaded Edge Model":
                    st.markdown("### 🔍 Granular Architecture Inspection")
                    with st.expander("Expand to view Layer-by-Layer Output Maps and MACs"):
                        try:
                            granular_stats = summary(model, input_size=(1, 3, 224, 224), col_names=["input_size", "output_size", "num_params", "mult_adds"], col_width=18, verbose=0)
                            st.code(str(granular_stats), language="text")
                        except Exception as e: st.warning("Granular trace disabled.")

                    st.markdown("### 📐 Spatial Dimension Extraction")
                    with st.expander("Expand to view Spatial Hyperparameters (Kernels, Strides, Padding)"):
                        try:
                            stats = summary(model, input_size=(1, 3, 224, 224), verbose=0)
                            spatial_data = [{"Layer Type": l.class_name, "Input": str(l.input_size), "Output": str(l.output_size), "Kernel": str(getattr(l.module, 'kernel_size', '-')), "Stride": str(getattr(l.module, 'stride', '-')), "Padding": str(getattr(l.module, 'padding', '-'))} for l in stats.summary_list if hasattr(l.module, 'kernel_size')]
                            if spatial_data: st.dataframe(pd.DataFrame(spatial_data), use_container_width=True, hide_index=True)
                        except Exception as e: st.warning("Spatial trace disabled.")

                elif source_type == "Upload ONNX File" and onnx_ops_df is not None:
                    st.markdown("### 🧩 Custom ONNX Graph Parsing")
                    with st.expander(f"Expand to view exact layers inside {display_name}"):
                        st.dataframe(onnx_ops_df, use_container_width=True, hide_index=True)

                st.divider()
                st.markdown("### 📋 Export PPA Scorecard")
                result_data = {
                    "hardware": {"tops": hw_tops, "bw": hw_bw, "clk": hw_clock, "pwr": hw_power_base, "l1": hw_l1_cache, "l2": hw_l2_cache},
                    "limits": {"mem": max_memory_mb, "lat": max_latency_ms, "pwr": max_power_w, "area": max_area_mm2},
                    "metrics": [
                        {"name": "Model", "value": display_name, "status": "INFO"},
                        {"name": "Precision", "value": quant_key, "status": "INFO"},
                        {"name": "Memory Required", "value": f"{total_memory_mb:.1f} MB", "status": "PASS" if total_memory_mb <= max_memory_mb else "FAIL"},
                        {"name": "Silicon Area", "value": f"{est_area_mm2:.1f} mm²", "status": "PASS" if est_area_mm2 <= max_area_mm2 else "FAIL"},
                        {"name": "Latency", "value": f"{latency_ms:.1f} ms", "status": "PASS" if latency_ms <= max_latency_ms else "FAIL"},
                        {"name": "Power Draw", "value": f"{power_w:.2f} W", "status": "PASS" if power_w <= max_power_w else "FAIL"},
                    ],
                    "verdict": "✓ DEPLOYMENT APPROVED" if issues == 0 else f"✗ CONSTRAINTS VIOLATED ({issues} checks failed)"
                }
                st.download_button(label="📥 Download PPA Scorecard (PDF)", data=generate_pdf_scorecard(result_data), file_name=f"ppa_scorecard_{display_name}_{quant_key}.pdf", mime="application/pdf")

# ==========================================
# BRANCH B: TRADITIONAL ML (CPU TARGET)
# ==========================================

else:
    st.info("💡 **Routing Workload to CPU:** Traditional ML models mapped to Edge CPU with cache-miss penalty modeling.")
    ml_model_type = st.selectbox("Choose a Traditional ML Architecture:", ["Decision Tree (CART)", "Random Forest (Ensemble)", "XGBoost (Gradient Boosting)", "Support Vector Machine (RBF Kernel)"])

    col1, col2, col3 = st.columns(3)
    num_features = col1.number_input("Number of Input Features", value=20, min_value=1)

    if ml_model_type in ["Random Forest (Ensemble)", "XGBoost (Gradient Boosting)"]:
        n_estimators = col2.slider("Number of Trees", 10, 500, 100)
        max_depth = col3.slider("Max Tree Depth", 2, 50, 10)
    elif ml_model_type == "Decision Tree (CART)":
        n_estimators = 1
        max_depth = col2.slider("Max Tree Depth", 2, 50, 15)
    elif ml_model_type == "Support Vector Machine (RBF Kernel)":
        n_support_vectors = col2.number_input("Number of Support Vectors", value=1500, min_value=10)

    if st.button("Run CPU Architecture Simulation", key="cpu_sim"):
        with st.spinner("Calculating CPU execution heuristics..."):
            if "Tree" in ml_model_type or "Forest" in ml_model_type or "XGBoost" in ml_model_type:
                total_memory_mb = (((2 ** max_depth) - 1) * n_estimators * 16) / (1024 * 1024)
                inference_ops = max_depth * n_estimators
                complexity = f"O({n_estimators} × {max_depth})" if n_estimators > 1 else f"O({max_depth})"
            elif "Support Vector" in ml_model_type:
                total_memory_mb = (n_support_vectors * num_features * 4) / (1024 * 1024)
                inference_ops = n_support_vectors * num_features
                complexity = f"O({n_support_vectors} × {num_features})"

            cache_result = cache_miss_latency(total_memory_mb, inference_ops, hw_clock, hw_l1_cache, hw_l2_cache)
            latency_ms = cache_result["latency_s"] * 1000
            throughput_fps = 1000 / latency_ms if latency_ms > 0 else 0
            power_w = hw_power_base + (0.05 * total_memory_mb)
            est_area_mm2 = total_memory_mb * 1.5

            st.divider()
            st.markdown(f"### 📊 Edge CPU Analysis: `{ml_model_type}`")

            c1, c2, c3, c4 = st.columns(4)
            c1.metric("Inference Operations", f"{inference_ops:,} Ops")
            c2.metric("Time Complexity", complexity)
            c3.metric("Cache Hit Level", cache_result["cache_hit_level"])
            c4.metric("Cache-Miss Penalty", f"{cache_result['miss_penalty_pct']:.0f}% of latency")

            st.markdown("#### 💾 Physical Footprint")
            c1, c2 = st.columns(2)
            c1.metric("Required RAM", format_memory(total_memory_mb), delta=f"Limit: {max_memory_mb} MB", delta_color="inverse" if total_memory_mb > max_memory_mb else "normal")
            c2.metric("SRAM Area (Est)", format_area(est_area_mm2), delta=f"Limit: {max_area_mm2} mm²", delta_color="inverse" if est_area_mm2 > max_area_mm2 else "normal")

            st.markdown("#### ⏱️ Execution Metrics")
            c1, c2, c3 = st.columns(3)
            c1.metric("Est. Latency", f"{latency_ms:,.2f} ms", delta=f"Limit: {max_latency_ms} ms", delta_color="inverse" if latency_ms > max_latency_ms else "normal")
            c2.metric("Inference / Sec", f"{throughput_fps:,.1f} FPS")
            c3.metric("Power Draw", format_power(power_w), delta=f"Limit: {max_power_w} W", delta_color="inverse" if power_w > max_power_w else "normal")

            # RESTORED: CPU Actionable Optimization Engine
            st.divider()
            st.markdown("### 🛠️ CPU Hardware-Aware Optimization")
            issues = 0

            if total_memory_mb > max_memory_mb:
                st.error(f"🚨 **Memory Overflow:** The tree depth or ensemble size requires {total_memory_mb:.1f} MB, exceeding the allocated {max_memory_mb} MB Edge RAM.")
                st.info("💡 **Action:** Restrict `max_depth` (pruning) or reduce `n_estimators`. Memory for tree algorithms grows exponentially ($2^{depth}$).")
                issues += 1

            if latency_ms > max_latency_ms:
                st.error(f"🚨 **Latency Failure:** The CPU pipeline is stalling due to excessive conditional branching ({latency_ms:.1f} ms > {max_latency_ms} ms).")
                if cache_result["cache_hit_level"] == "DRAM":
                    st.warning(f"⚠️ **Memory Locality Issue:** Cache-miss penalty is accounting for {cache_result['miss_penalty_pct']:.0f}% of your total latency. The model is too large for L2 and is thrashing main DRAM.")
                st.info("💡 **Action:** Convert the Random Forest into a single compiled C-header file using TinyML tools like `micromlgen` or `treelite` to remove Python runtime overhead.")
                issues += 1

            if power_w > max_power_w:
                st.error("🚨 **Power Failure:** Power consumption exceeds the physical budget of the device.")
                st.info("💡 **Action:** Reduce model size or apply dynamic voltage and frequency scaling (DVFS) to the Edge CPU.")
                issues += 1

            if issues == 0:
                st.success("✅ **CPU Deployment Approved:** The classical model satisfies all hardware constraints.")
            else:
                st.error(f"🚨 **Constraints Violated:** Failed {issues} hardware checks.")

            st.divider()
            st.markdown("### 📋 Export PPA Scorecard")
            result_data = {
                "hardware": {"tops": hw_tops, "bw": hw_bw, "clk": hw_clock, "pwr": hw_power_base, "l1": hw_l1_cache, "l2": hw_l2_cache},
                "limits": {"mem": max_memory_mb, "lat": max_latency_ms, "pwr": max_power_w, "area": max_area_mm2},
                "metrics": [
                    {"name": "Model Type", "value": ml_model_type, "status": "INFO"},
                    {"name": "Required RAM", "value": f"{total_memory_mb:.1f} MB", "status": "PASS" if total_memory_mb <= max_memory_mb else "FAIL"},
                    {"name": "Latency", "value": f"{latency_ms:.2f} ms", "status": "PASS" if latency_ms <= max_latency_ms else "FAIL"},
                    {"name": "Power Draw", "value": f"{power_w:.2f} W", "status": "PASS" if power_w <= max_power_w else "FAIL"},
                ],
                "verdict": "✓ CPU DEPLOYMENT APPROVED" if issues == 0 else f"✗ CONSTRAINTS VIOLATED ({issues} checks failed)"
            }
            st.download_button(label="📥 Download PPA Scorecard (PDF)", data=generate_pdf_scorecard(result_data), file_name=f"ppa_scorecard_{ml_model_type.split()[0]}.pdf", mime="application/pdf")

Overwriting app.py


##**Step 3: Background Process Manager**

In [6]:
# Stop previous background tasks
!pkill -f streamlit
!pkill -f ngrok
# Force pyngrok to terminate all active sessions
from pyngrok import ngrok
ngrok.kill()

##**Step 4: Public Deployment**

In [7]:
!pip install -q pyngrok

from pyngrok import ngrok

#Please replace this auth token , with the respective user's NGROK Token.
NGROK_TOKEN = "3CJPltDEn19sqyDgBLJsslIXcOk_4F4NeFmBfG3aqEJYYvdEY"
ngrok.set_auth_token(NGROK_TOKEN)

# Create a tunnel to Streamlit's default port (8501)
public_url = ngrok.connect(8501)
print(f" Click this link to open your IDE: {public_url}")

# Run the Streamlit app
!streamlit run app.py &>/content/logs.txt &


 Click this link to open your IDE: NgrokTunnel: "https://serotonin-carnage-rice.ngrok-free.dev" -> "http://localhost:8501"
